# Setup

In [3]:
from __future__ import annotations

from datetime import date
from html.parser import HTMLParser
from pathlib import Path
from urllib.error import HTTPError
from urllib.parse import urljoin, urlparse
from urllib.request import urlopen, urlretrieve
import shutil
import tarfile

import pandas as pd
import xarray as xr

## Configuration

In [4]:
ARCHIVE_BROWSER_URL = "https://archive.data.noaa.gov/satellite-spaceweather#SWPC/Models/ENLIL/swpc_wsaenlil_bkg/"
BASE_URL = "https://data.ngdc.noaa.gov/earth-science-services/models/space-weather/wsa-enlil"

def time_label(t):
    return str(pd.Timestamp(t).tz_localize(None)).replace(":", "-")


def shock_label(tShock):
    return time_label(tShock)


def resolve_run_interval(run_mode, tShock, custom_t1, custom_t2, window_hours=12):
    if run_mode == "shock":
        center = pd.to_datetime(tShock, utc=True)
        half_window = pd.Timedelta(hours=window_hours / 2)
        return center - half_window, center + half_window, center
    if run_mode == "custom":
        return pd.to_datetime(custom_t1, utc=True), pd.to_datetime(custom_t2, utc=True), None
    raise ValueError(f"Unsupported RUN_MODE: {run_mode!r}")


def resolve_output_dir(run_mode, tShock, t_start, t_end, base_dir="Data"):
    base_dir = Path(base_dir)
    if run_mode == "shock":
        return base_dir / shock_label(tShock)
    if run_mode == "custom":
        return base_dir / "Custom" / f"{time_label(t_start)}-{time_label(t_end)}"
    raise ValueError(f"Unsupported RUN_MODE: {run_mode!r}")



In [ ]:
RUN_MODE = "custom"  # "shock" or "custom"
tShock = pd.to_datetime("2022-11-24T19:10:00Z")
CUSTOM_T1 = pd.to_datetime("2017-01-01T00:00:00Z")
CUSTOM_T2 = pd.to_datetime("2026-01-01T00:00:00Z")
SHOCK_WINDOW_HOURS = 12

t_start, t_end, tShock = resolve_run_interval(
    RUN_MODE,
    tShock,
    CUSTOM_T1,
    CUSTOM_T2,
    window_hours=SHOCK_WINDOW_HOURS,
)


In [6]:

OUTPUT_DIR = resolve_output_dir(RUN_MODE, tShock, t_start, t_end)

RUN_DATE = t_start.date()
CYCLE = "0000"

DOWNLOAD_DIR = OUTPUT_DIR / "ENLIL" / "tars"
EXTRACT_DIR = OUTPUT_DIR / "ENLIL" / "extracted"
PARQUET_DATASET_DIR = OUTPUT_DIR / "ENLIL"
MERGED_PARQUET_DIR = OUTPUT_DIR / "ENLIL"

RESAMPLE_FREQ = "5min"
CLEANUP_TARS = True
CLEANUP_EXTRACTED = True
CLEANUP_DAILY_PARQUETS_AFTER_MERGE = True

MERGE_MODE = "all"  # "all", "yearly", or "monthly"
MERGE_T1 = t_start
MERGE_T2 = t_end

OBSERVER_PREFIXES = ("Earth", "STEREO_A")
OBSERVER_FIELDS = ("V1", "V2", "V3", "B1", "B2", "B3")


# Helper Functions

In [7]:
class LinkParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.hrefs = []

    def handle_starttag(self, tag, attrs):
        if tag != "a":
            return
        attrs = dict(attrs)
        href = attrs.get("href")
        if href is not None:
            self.hrefs.append(href)


def month_url(year, month):
    return f"{BASE_URL}/{year:04d}/{month:02d}/"


def archive_name(run_date, cycle=CYCLE):
    return f"swpc_wsaenlil_bkg_{run_date:%Y%m%d}_{cycle}.tar.gz"


def archive_url(run_date, cycle=CYCLE):
    return urljoin(month_url(run_date.year, run_date.month), archive_name(run_date, cycle))


def list_month_archives(year, month):
    url = month_url(year, month)
    html = urlopen(url).read().decode("utf-8")
    parser = LinkParser()
    parser.feed(html)
    urls = [urljoin(url, href) for href in parser.hrefs if href.endswith(".tar.gz")]
    return sorted(urls)

## Files and Archives

In [8]:
def download_archive(url, download_dir=DOWNLOAD_DIR, overwrite=False):
    download_dir = Path(download_dir)
    download_dir.mkdir(parents=True, exist_ok=True)

    filename = Path(urlparse(url).path).name
    path = download_dir / filename
    if path.exists() and not overwrite:
        return path

    urlretrieve(url, path)
    return path


def extract_archive(tar_path, extract_dir=EXTRACT_DIR, overwrite=False):
    tar_path = Path(tar_path)
    extract_dir = Path(extract_dir)
    destination = extract_dir / tar_path.name.removesuffix(".tar.gz")

    if destination.exists() and overwrite:
        shutil.rmtree(destination)

    if not destination.exists():
        destination.mkdir(parents=True, exist_ok=True)
        with tarfile.open(tar_path, "r:gz") as archive:
            archive.extractall(destination, filter="data")

    return destination


def find_netcdf_files(root):
    root = Path(root)
    return sorted(root.rglob("*.nc"))


def open_enlil_netcdf(nc_path):
    return xr.open_dataset(nc_path, decode_timedelta=True)

## Resampling and Daily Parquets

In [9]:
def observer_columns(prefixes=OBSERVER_PREFIXES, fields=OBSERVER_FIELDS):
    return [f"{prefix}_{field}" for prefix in prefixes for field in fields]


def enlil_run_id(ds, cycle=CYCLE):
    ref_time = pd.Timestamp(ds.attrs["REFDATE_CAL"])
    return f"swpc_wsaenlil_bkg_{ref_time:%Y%m%d}_{cycle}"


def absolute_earth_time(ds):
    ref_time = pd.Timestamp(ds.attrs["REFDATE_CAL"], tz="UTC")
    return ref_time + pd.to_timedelta(ds["Earth_TIME"].values)


def observer_dataframe(ds, nc_path=None, archive_url=None):
    frame = ds[observer_columns()].to_dataframe().reset_index(drop=True)
    frame.insert(0, "time", absolute_earth_time(ds))
    frame.insert(1, "run_id", enlil_run_id(ds))
    frame.insert(2, "source_netcdf", "" if nc_path is None else str(nc_path))
    frame.insert(3, "source_archive_url", "" if archive_url is None else archive_url)
    return frame


def resample_observer_dataframe(frame, freq=RESAMPLE_FREQ):
    value_columns = observer_columns()
    metadata = frame[["run_id", "source_netcdf", "source_archive_url"]].iloc[0]
    resampled = (
        frame.set_index("time")[value_columns]
        .sort_index()
        .resample(freq, label="left", closed="left")
        .mean()
        .dropna(how="all")
        .reset_index()
    )
    resampled.insert(1, "run_id", metadata["run_id"])
    resampled.insert(2, "source_netcdf", metadata["source_netcdf"])
    resampled.insert(3, "source_archive_url", metadata["source_archive_url"])
    return resampled


def resample_enlil_observers(ds, nc_path=None, archive_url=None, freq=RESAMPLE_FREQ):
    frame = observer_dataframe(ds, nc_path=nc_path, archive_url=archive_url)
    return resample_observer_dataframe(frame, freq=freq)


def expected_run_id(run_date, cycle=CYCLE):
    return f"swpc_wsaenlil_bkg_{run_date:%Y%m%d}_{cycle}"


def expected_parquet_path(run_date, cycle=CYCLE, parquet_dir=PARQUET_DATASET_DIR):
    return Path(parquet_dir) / f"{expected_run_id(run_date, cycle=cycle)}.parquet"


def append_resampled_parquet(frame, parquet_dir=PARQUET_DATASET_DIR, overwrite_run=False):
    parquet_dir = Path(parquet_dir)
    parquet_dir.mkdir(parents=True, exist_ok=True)

    run_ids = frame["run_id"].drop_duplicates().tolist()
    if len(run_ids) != 1:
        raise ValueError(f"Expected one run_id, got {run_ids!r}")

    parquet_path = parquet_dir / f"{run_ids[0]}.parquet"
    ordered = frame.sort_values(["run_id", "time"])
    if parquet_path.exists() and not overwrite_run:
        existing = pd.read_parquet(parquet_path)
        ordered = (
            pd.concat([existing, ordered], ignore_index=True)
            .drop_duplicates(subset=["run_id", "time"])
            .sort_values(["run_id", "time"])
        )

    ordered.to_parquet(parquet_path, index=False)
    return parquet_path

## Batch Processing

In [10]:
def daily_run_dates(start, end):
    start_day = pd.Timestamp(start).tz_convert("UTC").normalize()
    end_time = pd.Timestamp(end).tz_convert("UTC")
    return [ts.date() for ts in pd.date_range(start_day, end_time, freq="D") if ts < end_time]


def process_enlil_run(
    run_date,
    cycle=CYCLE,
    download_dir=DOWNLOAD_DIR,
    extract_dir=EXTRACT_DIR,
    parquet_dir=PARQUET_DATASET_DIR,
    overwrite_download=False,
    overwrite_extract=False,
    overwrite_run=False,
    cleanup_tar=CLEANUP_TARS,
    cleanup_extracted=CLEANUP_EXTRACTED,
):
    url = archive_url(run_date, cycle=cycle)
    parquet_path = expected_parquet_path(run_date, cycle=cycle, parquet_dir=parquet_dir)
    if parquet_path.exists() and not overwrite_run:
        return {
            "run_date": run_date,
            "archive_url": url,
            "parquet_path": parquet_path,
            "status": "exists",
            "rows": pd.read_parquet(parquet_path, columns=["time"]).shape[0],
        }

    tar_path = None
    extracted_dir = None
    nc_path = None
    ds = None
    try:
        tar_path = download_archive(url, download_dir=download_dir, overwrite=overwrite_download)
        extracted_dir = extract_archive(tar_path, extract_dir=extract_dir, overwrite=overwrite_extract)
        nc_paths = find_netcdf_files(extracted_dir)
        if len(nc_paths) != 1:
            raise ValueError(f"Expected one NetCDF under {extracted_dir}, got {len(nc_paths)}: {nc_paths}")

        nc_path = nc_paths[0]
        ds = open_enlil_netcdf(nc_path)
        resampled = resample_enlil_observers(ds, nc_path=nc_path, archive_url=url)
        parquet_path = append_resampled_parquet(resampled, parquet_dir=parquet_dir, overwrite_run=overwrite_run)

        return {
            "run_date": run_date,
            "archive_url": url,
            "tar_path": tar_path,
            "extracted_dir": extracted_dir,
            "nc_path": nc_path,
            "parquet_path": parquet_path,
            "status": "ok",
            "rows": len(resampled),
            "t_min": resampled["time"].min(),
            "t_max": resampled["time"].max(),
        }
    finally:
        if ds is not None:
            ds.close()
        if cleanup_extracted and extracted_dir is not None and extracted_dir.exists():
            shutil.rmtree(extracted_dir)
        if cleanup_tar and tar_path is not None and tar_path.exists():
            tar_path.unlink()


def process_enlil_interval(
    start,
    end,
    cycle=CYCLE,
    download_dir=DOWNLOAD_DIR,
    extract_dir=EXTRACT_DIR,
    parquet_dir=PARQUET_DATASET_DIR,
    overwrite_run=False,
    cleanup_tar=CLEANUP_TARS,
    cleanup_extracted=CLEANUP_EXTRACTED,
    continue_on_error=True,
):
    records = []
    for run_date in daily_run_dates(start, end):
        print(f"Processing {archive_name(run_date, cycle=cycle)}")
        try:
            record = process_enlil_run(
                run_date,
                cycle=cycle,
                download_dir=download_dir,
                extract_dir=extract_dir,
                parquet_dir=parquet_dir,
                overwrite_run=overwrite_run,
                cleanup_tar=cleanup_tar,
                cleanup_extracted=cleanup_extracted,
            )
        except HTTPError as exc:
            if exc.code != 404 and not continue_on_error:
                raise
            record = {
                "run_date": run_date,
                "archive_url": archive_url(run_date, cycle=cycle),
                "status": "missing" if exc.code == 404 else "http_error",
                "error_type": type(exc).__name__,
                "error": str(exc),
            }
            print(f"Skipping {archive_name(run_date, cycle=cycle)}: HTTP {exc.code}")
        except Exception as exc:
            if not continue_on_error:
                raise
            record = {
                "run_date": run_date,
                "archive_url": archive_url(run_date, cycle=cycle),
                "status": "error",
                "error_type": type(exc).__name__,
                "error": str(exc),
            }
            print(f"Skipping {archive_name(run_date, cycle=cycle)}: {type(exc).__name__}: {exc}")
        records.append(record)
    return pd.DataFrame.from_records(records)

# Single Run Workflow

Use these cells to inspect one archive, download it, open the NetCDF, resample the observer series, and write a daily Parquet.

In [11]:
archive_index_url = month_url(RUN_DATE.year, RUN_DATE.month)
archive_index_url

'https://data.ngdc.noaa.gov/earth-science-services/models/space-weather/wsa-enlil/2018/02/'

In [12]:
available_archives = list_month_archives(RUN_DATE.year, RUN_DATE.month)
available_archives[:5], len(available_archives)

(['https://data.ngdc.noaa.gov/earth-science-services/models/space-weather/wsa-enlil/2018/02/swpc_wsaenlil_bkg_20180201_0000.tar.gz',
  'https://data.ngdc.noaa.gov/earth-science-services/models/space-weather/wsa-enlil/2018/02/swpc_wsaenlil_bkg_20180202_0000.tar.gz',
  'https://data.ngdc.noaa.gov/earth-science-services/models/space-weather/wsa-enlil/2018/02/swpc_wsaenlil_bkg_20180203_0000.tar.gz',
  'https://data.ngdc.noaa.gov/earth-science-services/models/space-weather/wsa-enlil/2018/02/swpc_wsaenlil_bkg_20180204_0000.tar.gz',
  'https://data.ngdc.noaa.gov/earth-science-services/models/space-weather/wsa-enlil/2018/02/swpc_wsaenlil_bkg_20180205_0000.tar.gz'],
 31)

## Select Archive

In [13]:
selected_archive_url = archive_url(RUN_DATE, CYCLE)
selected_archive_url

'https://data.ngdc.noaa.gov/earth-science-services/models/space-weather/wsa-enlil/2018/02/swpc_wsaenlil_bkg_20180201_0000.tar.gz'

Download the selected archive.

In [14]:
tar_path = download_archive(selected_archive_url, DOWNLOAD_DIR)
tar_path

PosixPath('Data/Custom/2018-02-01 00-00-00-2018-07-01 00-00-00/ENLIL/tars/swpc_wsaenlil_bkg_20180201_0000.tar.gz')

Extract the downloaded archive.

In [15]:
extracted_dir = extract_archive(tar_path, EXTRACT_DIR)
extracted_dir

PosixPath('Data/Custom/2018-02-01 00-00-00-2018-07-01 00-00-00/ENLIL/extracted/swpc_wsaenlil_bkg_20180201_0000')

## Open and Inspect NetCDF

In [16]:
nc_paths = find_netcdf_files(extracted_dir)
nc_paths

[PosixPath('Data/Custom/2018-02-01 00-00-00-2018-07-01 00-00-00/ENLIL/extracted/swpc_wsaenlil_bkg_20180201_0000/wsa_enlil.latest.suball.nc')]

In [17]:
ds = open_enlil_netcdf(nc_paths[0])
ds

<xarray.Dataset> Size: 206MB
Dimensions:                   (x: 512, y: 60, z: 180, t: 169, earth_t: 10489,
                               fld_step: 1536, fld_directions: 2)
Dimensions without coordinates: x, y, z, t, earth_t, fld_step, fld_directions
Data variables: (12/83)
    x_coord                   (x) float32 2kB ...
    y_coord                   (y) float32 240B ...
    z_coord                   (z) float32 720B ...
    time                      (t) timedelta64[ns] 1kB ...
    dd12_3d                   (t, y, x) int16 10MB ...
    vv12_3d                   (t, y, x) int16 10MB ...
    ...                        ...
    STEREO_B_FLD_V1           (t, fld_step, fld_directions) float32 2MB ...
    STEREO_B_FLD_V2           (t, fld_step, fld_directions) float32 2MB ...
    STEREO_B_FLD_V3           (t, fld_step, fld_directions) float32 2MB ...
    STEREO_B_FLD_B1           (t, fld_step, fld_directions) float32 2MB ...
    STEREO_B_FLD_B2           (t, fld_step, fld_directions) float32 2MB ...
    STEREO_B_FLD_B3           (t, fld_step, fld_directions) float32 2MB ...
Attributes: (12/20)
    REFDATE_MJD:              58150.0
    REFDATE_CAL:              2018-02-01T00:00:00
    OBSDATE_MJD:              58149.883
    OBSDATE_CAL:              2018-01-31T21:12:48
    program:                  enlil
    enlil_version:            2.6
    ...                       ...
    cyc:                      00
    datafile_format_version:  1.0
    summary:                  Standard Enlil model output file as utilized by...
    cal_min:                  -32768.0
    cal_range:                65535.0
    calibration_explanation:  All parameters whose names begin with 'uncalibr...

## Resample Observer Series

In [18]:
resampled = resample_enlil_observers(ds, nc_path=nc_paths[0], archive_url=selected_archive_url)
resampled.head(), resampled.tail(), resampled.shape

(                       time                           run_id  \
 0 2018-01-17 00:00:00+00:00  swpc_wsaenlil_bkg_20180201_0000   
 1 2018-01-17 00:05:00+00:00  swpc_wsaenlil_bkg_20180201_0000   
 2 2018-01-17 00:10:00+00:00  swpc_wsaenlil_bkg_20180201_0000   
 3 2018-01-17 00:15:00+00:00  swpc_wsaenlil_bkg_20180201_0000   
 4 2018-01-17 00:20:00+00:00  swpc_wsaenlil_bkg_20180201_0000   
 
                                        source_netcdf  \
 0  Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...   
 1  Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...   
 2  Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...   
 3  Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...   
 4  Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...   
 
                                   source_archive_url     Earth_V1  Earth_V2  \
 0  https://data.ngdc.noaa.gov/earth-science-servi...  399212.6875  0.080873   
 1  https://data.ngdc.noaa.gov/earth-science-servi...  399215.3750  0.411138   
 2  https://data.ngdc.no

Write the resampled run to its daily Parquet.

In [19]:
parquet_path = append_resampled_parquet(resampled, PARQUET_DATASET_DIR)
parquet_path

PosixPath('Data/Custom/2018-02-01 00-00-00-2018-07-01 00-00-00/ENLIL/swpc_wsaenlil_bkg_20180201_0000.parquet')

# Full Interval Processing

By default this deletes each downloaded `.tar.gz` and extracted run directory after that run has been written to Parquet. Set `CLEANUP_TARS` or `CLEANUP_EXTRACTED` to `False` in the configuration cell to keep those intermediates.

In [20]:
run_dates = daily_run_dates(t_start, t_end)
run_dates[:3], run_dates[-3:], len(run_dates)

([datetime.date(2018, 2, 1),
  datetime.date(2018, 2, 2),
  datetime.date(2018, 2, 3)],
 [datetime.date(2018, 6, 28),
  datetime.date(2018, 6, 29),
  datetime.date(2018, 6, 30)],
 150)

In [21]:
batch_manifest = process_enlil_interval(
    t_start,
    t_end,
    cycle=CYCLE,
    download_dir=DOWNLOAD_DIR,
    extract_dir=EXTRACT_DIR,
    parquet_dir=PARQUET_DATASET_DIR,
    cleanup_tar=CLEANUP_TARS,
    cleanup_extracted=CLEANUP_EXTRACTED,
)
batch_manifest

Processing swpc_wsaenlil_bkg_20180201_0000.tar.gz
Processing swpc_wsaenlil_bkg_20180202_0000.tar.gz
Processing swpc_wsaenlil_bkg_20180203_0000.tar.gz
Processing swpc_wsaenlil_bkg_20180204_0000.tar.gz
Processing swpc_wsaenlil_bkg_20180205_0000.tar.gz
Processing swpc_wsaenlil_bkg_20180206_0000.tar.gz
Processing swpc_wsaenlil_bkg_20180207_0000.tar.gz
Processing swpc_wsaenlil_bkg_20180208_0000.tar.gz
Processing swpc_wsaenlil_bkg_20180209_0000.tar.gz
Processing swpc_wsaenlil_bkg_20180210_0000.tar.gz
Processing swpc_wsaenlil_bkg_20180211_0000.tar.gz
Processing swpc_wsaenlil_bkg_20180212_0000.tar.gz
Processing swpc_wsaenlil_bkg_20180213_0000.tar.gz
Processing swpc_wsaenlil_bkg_20180214_0000.tar.gz
Processing swpc_wsaenlil_bkg_20180215_0000.tar.gz
Processing swpc_wsaenlil_bkg_20180216_0000.tar.gz
Processing swpc_wsaenlil_bkg_20180217_0000.tar.gz
Processing swpc_wsaenlil_bkg_20180218_0000.tar.gz
Processing swpc_wsaenlil_bkg_20180219_0000.tar.gz
Processing swpc_wsaenlil_bkg_20180220_0000.tar.gz


,run_date,archive_url,parquet_path,status,rows,tar_path,extracted_dir,nc_path,t_min,t_max,error_type,error
0,2018-02-01,https://data.ngdc.noaa.gov/earth-science-servi...,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,exists,5761.0,NaN,NaN,NaN,NaT,NaT,NaN,NaN
1,2018-02-02,https://data.ngdc.noaa.gov/earth-science-servi...,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,ok,5761.0,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,2018-01-18 00:00:00+00:00,2018-02-07 00:00:00+00:00,NaN,NaN
2,2018-02-03,https://data.ngdc.noaa.gov/earth-science-servi...,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,ok,5761.0,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,2018-01-19 00:00:00+00:00,2018-02-08 00:00:00+00:00,NaN,NaN
3,2018-02-04,https://data.ngdc.noaa.gov/earth-science-servi...,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,ok,5761.0,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,2018-01-20 00:00:00+00:00,2018-02-09 00:00:00+00:00,NaN,NaN
4,2018-02-05,https://data.ngdc.noaa.gov/earth-science-servi...,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,ok,5761.0,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,2018-01-21 00:00:00+00:00,2018-02-10 00:00:00+00:00,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
145,2018-06-26,https://data.ngdc.noaa.gov/earth-science-servi...,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,ok,5761.0,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,2018-06-11 00:00:00+00:00,2018-07-01 00:00:00+00:00,NaN,NaN
146,2018-06-27,https://data.ngdc.noaa.gov/earth-science-servi...,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,ok,5761.0,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,2018-06-12 00:00:00+00:00,2018-07-02 00:00:00+00:00,NaN,NaN
147,2018-06-28,https://data.ngdc.noaa.gov/earth-science-servi...,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,ok,5761.0,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,2018-06-13 00:00:00+00:00,2018-07-03 00:00:00+00:00,NaN,NaN
148,2018-06-29,https://data.ngdc.noaa.gov/earth-science-servi...,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,ok,5761.0,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...,2018-06-14 00:00:00+00:00,2018-07-04 00:00:00+00:00,NaN,NaN


# Merge Outputs

Merge daily run Parquets over `MERGE_T1` to `MERGE_T2`. `MERGE_MODE` controls whether the output is one file for the whole interval, one file per year, or one file per month. Daily Parquet cleanup is optional and defaults to off.

In [22]:
def merge_period_windows(start, end, mode=MERGE_MODE):
    start_time = pd.Timestamp(start).tz_convert("UTC")
    end_time = pd.Timestamp(end).tz_convert("UTC")
    if mode == "all":
        label = f"{time_label(start_time)}-{time_label(end_time)}"
        return [(label, start_time, end_time)]

    if mode == "monthly":
        current = pd.Timestamp(year=start_time.year, month=start_time.month, day=1, tz="UTC")
        offset = pd.DateOffset(months=1)
        label_format = "%Y-%m"
    elif mode == "yearly":
        current = pd.Timestamp(year=start_time.year, month=1, day=1, tz="UTC")
        offset = pd.DateOffset(years=1)
        label_format = "%Y"
    else:
        raise ValueError(f"Unsupported MERGE_MODE: {mode!r}")

    windows = []
    while current < end_time:
        next_time = current + offset
        window_start = max(current, start_time)
        window_end = min(next_time, end_time)
        if window_start < window_end:
            windows.append((current.strftime(label_format), window_start, window_end))
        current = next_time
    return windows


def daily_parquet_paths_for_interval(start, end, cycle=CYCLE, parquet_dir=PARQUET_DATASET_DIR):
    paths = [
        expected_parquet_path(run_date, cycle=cycle, parquet_dir=parquet_dir)
        for run_date in daily_run_dates(start, end)
    ]
    return [path for path in paths if path.exists()]


def merge_daily_parquets(
    start=MERGE_T1,
    end=MERGE_T2,
    mode=MERGE_MODE,
    cycle=CYCLE,
    parquet_dir=PARQUET_DATASET_DIR,
    out_dir=MERGED_PARQUET_DIR,
    cleanup_daily=CLEANUP_DAILY_PARQUETS_AFTER_MERGE,
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    records = []

    for label, window_start, window_end in merge_period_windows(start, end, mode=mode):
        paths = daily_parquet_paths_for_interval(window_start, window_end, cycle=cycle, parquet_dir=parquet_dir)
        out_path = out_dir / f"ENLIL.parquet"
        if not paths:
            records.append({
                "label": label,
                "t_start": window_start,
                "t_end": window_end,
                "status": "no_files",
                "daily_files": 0,
                "rows": 0,
                "out_path": out_path,
            })
            continue

        merged = (
            pd.concat((pd.read_parquet(path) for path in paths), ignore_index=True)
            .drop_duplicates(subset=["run_id", "time"])
            .sort_values(["run_id", "time"])
        )
        merged.to_parquet(out_path, index=False)

        if cleanup_daily:
            for path in paths:
                path.unlink()

        records.append({
            "label": label,
            "t_start": window_start,
            "t_end": window_end,
            "status": "merged",
            "daily_files": len(paths),
            "rows": len(merged),
            "out_path": out_path,
        })

    return pd.DataFrame.from_records(records)

In [23]:
merge_manifest = merge_daily_parquets(
    start=MERGE_T1,
    end=MERGE_T2,
    mode=MERGE_MODE,
    cycle=CYCLE,
    parquet_dir=PARQUET_DATASET_DIR,
    out_dir=MERGED_PARQUET_DIR,
    cleanup_daily=CLEANUP_DAILY_PARQUETS_AFTER_MERGE,
)
merge_manifest

,label,t_start,t_end,status,daily_files,rows,out_path
0,2018-02-01 00-00-00-2018-07-01 00-00-00,2018-02-01 00:00:00+00:00,2018-07-01 00:00:00+00:00,merged,149,858389,Data/Custom/2018-02-01 00-00-00-2018-07-01 00-...


In [ ]:
!curl -d "WSA-ENLIL downloader finished" ntfy.sh/helio-n